<a href="https://colab.research.google.com/github/Koshalendar/Artificial-intelligence-/blob/main/First_order_logic__AI.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [7]:
def unify(expr1, expr2, theta=None):
    if theta is None:
        theta = {}
    if theta is False:
        return False
    if expr1 == expr2:
        return theta
    if isinstance(expr1, str) and expr1.islower():
        return unify_var(expr1, expr2, theta)
    if isinstance(expr2, str) and expr2.islower():
        return unify_var(expr2, expr1, theta)
    if isinstance(expr1, tuple) and isinstance(expr2, tuple):
        if len(expr1) != len(expr2):
            return False
        for e1, e2 in zip(expr1, expr2):
            theta = unify(e1, e2, theta)
        return theta
    return False

def unify_var(var, val, theta):
    if var in theta:
        return unify(theta[var], val, theta)
    if val in theta:
        return unify(var, theta[val], theta)
    new_theta = theta.copy()
    new_theta[var] = val
    return new_theta

def subst(theta, expr):
    if isinstance(expr, str):
        return theta.get(expr, expr)
    if isinstance(expr, tuple):
        return tuple(subst(theta, e) for e in expr)
    return expr

def match_premises(premises, kb, theta, index=0):
    if index == len(premises):
        return [theta]
    results = []
    p = premises[index]
    for fact in kb:
        if isinstance(fact, tuple) and len(fact) == len(p):
            new_theta = unify(p, fact, theta.copy())
            if new_theta is not False:
                results.extend(match_premises(premises, kb, new_theta, index + 1))
    return results

def fol_fc_ask(kb_rules, kb_facts, alpha):
    kb = list(kb_facts)
    counter = 0
    while True:
        new = []
        for premises, conclusion in kb_rules:
            counter += 1
            var_map = {}
            for p in premises:
                for v in p:
                    if isinstance(v, str) and v.islower():
                        var_map[v] = f"{v}_{counter}"
            for v in conclusion:
                if isinstance(v, str) and v.islower():
                    var_map[v] = f"{v}_{counter}"

            std_premises = tuple(tuple(var_map.get(e, e) for e in p) for p in premises)
            std_conclusion = tuple(var_map.get(e, e) for e in conclusion)

            thetas = match_premises(std_premises, kb, {})
            for theta in thetas:
                q_prime = subst(theta, std_conclusion)
                if q_prime not in kb and q_prime not in new:
                    new.append(q_prime)
                    # Print out intermediate inferences to show forward chaining in action
                    print(f"-> Inferred: {q_prime[0]}({q_prime[1]})")
                    phi = unify(q_prime, alpha)
                    if phi is not False:
                        return phi
        if not new:
            return False
        kb.extend(new)

rules = [
    ((('Fever', 'x'), ('SoreThroat', 'x')), ('InfectionSigns', 'x')),
    ((('InfectionSigns', 'x'), ('MuscleAches', 'x')), ('TestFlu', 'x'))
]

facts = []
patient_name = input("Enter patient identifier (e.g., patient_p): ").strip()

if input(f"Does {patient_name} have a fever? (y/n): ").strip().lower() in ["y", "yes"]:
    facts.append(('Fever', patient_name))

if input(f"Does {patient_name} have a sore throat? (y/n): ").strip().lower() in ["y", "yes"]:
    facts.append(('SoreThroat', patient_name))

if input(f"Does {patient_name} have muscle aches? (y/n): ").strip().lower() in ["y", "yes"]:
    facts.append(('MuscleAches', patient_name))

print("\n--- Forward Chaining Process ---")
query = ('TestFlu', patient_name)
result = fol_fc_ask(rules, facts, query)

print("\n--- Final Result ---")
if result is not False:
    print(f"SUCCESS: The forward chaining deduction holds true for {patient_name}.")
    print(f"CONCLUSION: Recommendation is to test {patient_name} for Influenza (the flu).")
else:
    print(f"FAILED: The rules cannot prove that {patient_name} needs a flu test based on current symptoms.")


Enter patient identifier (e.g., patient_p): p
Does p have a fever? (y/n): y
Does p have a sore throat? (y/n): y
Does p have muscle aches? (y/n): y

--- Forward Chaining Process ---
-> Inferred: InfectionSigns(p)
-> Inferred: TestFlu(p)

--- Final Result ---
SUCCESS: The forward chaining deduction holds true for p.
CONCLUSION: Recommendation is to test p for Influenza (the flu).
